In [91]:
## IMPORT LIBRARIES ----------------------------------------------------------------------------------------------------
#https://github.com/nnc-ufmg/circadipy/blob/main/src/circadipy/analysis_examples/intellicage/intellicage_analysis.ipynb
%matplotlib qt

%reload_ext autoreload
%autoreload 3

from circadipy import chrono_reader as chr  
import sys                                                                                                              # Import sys to add paths to libraries                                                                                                           # Import re to work with regular expressions
import glob                                                                                                             # Import glob to read files                                                                                                   # Import numpy to work with arrays and make calculations                                                                                            # Import time to measure time
import os                                                                                                               # Import path to work with paths                
import pandas as pd
from ranking_methods import build_all_proxies, evaluate_proxies
import numpy as np
from scipy import stats
from datetime import date, datetime, timedelta
from itertools import combinations
from scipy.stats import rankdata, spearmanr, kendalltau
from ranking_methods import rank_accuracy                                                                                   # Import pandas to work with dataframes
import warnings                                                                                                         # Import warnings to ignore warnings
warnings.filterwarnings('ignore')                                                                                       # Ignore warnings

## IMPORT CIRCADIPY ----------------------------------------------------------------------------------------------------

parent_path = os.path.dirname(os.path.dirname(os.getcwd()))
sys.path.append(parent_path)

## PCA Visualization - Dimensionality Reduction
from models import train_rf, train_gb, train_ridge, train_adaboost, train_extratrees, train_logistic
from summary import generate_summary_report
from util import  get_data_scaled, generate_features, generate_temporal_features, calculate_correlations, build_animal_protocols, get_sorted_animals_files, combine_all_features
from analysis_visualization import generate_methods_comparison, plot_animals_activity, plot_correlation, plot_pca_analysis, plot_cross_correlation, summary_visualization

In [ ]:
previous_data = True
best_feat = None   # None = auto-pick highest |ρ| feature from corr_df
#best_feat = 'ibi_median'
#named_combo = ['ibi_median', 'hourly_entropy', 'power_8h', 'bout_len_cv']

named_combo = None
#named_combo = ['rhythm_ratio_24_over_harm', 'daily_peak_std', 'night_frac', 'num_bouts_night']
if not previous_data:
    data_folder = "./data/dados_iniciais_estruturados"    
    individual_files = glob.glob(data_folder + "/**/*.zip", recursive=True)
    ranks = {
        "animal_1": 9,
        "animal_2": 8,
        "animal_3": 11,
        "animal_4": 10,
        "animal_5": 12,
        "animal_6": 1,
        "animal_7": 5,
        "animal_8": 6,
        "animal_9": 7,
        "animal_10": 2,
        "animal_11": 3,
        "animal_12": 4
    }

    
    output_folder = './results/test_results_2026_new_data'

    # Build date range between two dates (inclusive)
    start_date = date(2026, 3, 17)
    end_date   = date(2026, 3, 20)
    dates_to_keep = [(start_date + timedelta(days=i)).strftime("%Y-%m-%d")
                    for i in range((end_date - start_date).days + 1)]
    force_rewrite = False
    individual_files = glob.glob(data_folder + "/**/*.txt", recursive=True)

else:

    data_folder = "./data"                              
    individual_files = glob.glob(data_folder + "/dados_iniciais/**/*.txt", recursive=True)
    ranks = {
        "animal_5": 5,
        "animal_12": 8,
        "animal_9": 6,
        "animal_13": 4,
        "animal_15": 1,
        "animal_3": 3,
        "animal_8": 7,
        "animal_11": 2
    }

    
    output_folder = './results/test_results_2026_previous_data3'


    dates_to_keep = []
    force_rewrite = False

if force_rewrite:
    for file in individual_files:
        print(file)
        folder = file.split("/")[-1].split(".zip")[0]

        
        root_folder = os.path.join(data_folder, folder)
        os.makedirs(root_folder, exist_ok=True)
        a = chr.intellicage_unwrapper([file], root_folder=root_folder, sampling_interval = '30T')
    




if dates_to_keep:
    individual_files = [f for f in individual_files if f.split("/")[-2].split(" ")[0] in dates_to_keep]


In [93]:


#usando sum para concatenar os protocolos gera resultado melhor que o last
#Se eu quiser ler os protocolos eu preciso do zt_0_time, neste caso estou usando 20horas.
#Tambem precisariamos definir o labels dict, neste caso o que utilizariamos?
zt_0_time = 20  #Para gerar o actograma é usado o zt dado de quando a luz é acesa, que será as 20 horas. 
apply_filtering = True
labels_dict = {'cycle_types': ['LD'], 'test_labels': ['1_control_dl'], 'cycle_days': [1]}
animals = [int(k.split("_")[1]) for k in ranks.keys()]
animals_files = get_sorted_animals_files(individual_files, animals)
for k, v in animals_files.items():
    print(f"{k}: {v}")
animals_protocols, animals_by_day = build_animal_protocols(animals_files, apply_filtering=apply_filtering)


Animal 3: 18
Animal 5: 16
Animal 8: 18
Animal 9: 19
Animal 11: 20
Animal 12: 20
Animal 13: 20
Animal 15: 19
3: ['./data/unwrapped_data/2025-03-01_15.02.33/animal_3.txt', './data/unwrapped_data/2025-03-02_15.11.11/animal_3.txt', './data/unwrapped_data/2025-03-04_20.22.59/animal_3.txt', './data/unwrapped_data/2025-03-05_14.57.54/animal_3.txt', './data/unwrapped_data/2025-03-07_08.57.45/animal_3.txt', './data/unwrapped_data/2025-03-10_14.31.46/animal_3.txt', './data/unwrapped_data/2025-03-14_11.12.27/animal_3.txt', './data/unwrapped_data/2025-03-14_15.41.35/animal_3.txt', './data/unwrapped_data/2025-03-14_16.22.42/animal_3.txt', './data/unwrapped_data/2025-03-16_18.11.28/animal_3.txt', './data/unwrapped_data/2025-03-17_14.40.35/animal_3.txt', './data/unwrapped_data/2025-03-18_10.13.19/animal_3.txt', './data/unwrapped_data/2025-03-18_18.29.47/animal_3.txt', './data/unwrapped_data/2025-03-19_19.16.59/animal_3.txt', './data/unwrapped_data/2025-03-21_14.14.15/animal_3.txt', './data/unwrapped_

In [94]:
os.makedirs(output_folder, exist_ok=True)


output_path = f'{output_folder}/basic_features.csv'
features_df = generate_features(animals_protocols, output_path=output_path)


output_path = f'{output_folder}/temporal_features.csv'
temporal_df = generate_temporal_features(animals_protocols, output_path=output_path)

output_path = f'{output_folder}/all_features.csv'
all_features, feature_cols = combine_all_features(features_df, temporal_df, output_path=output_path)

# Derived additive features
y = []
for animal in all_features['animal'].tolist():
    y.append(ranks[animal])

X_scaled =  get_data_scaled(all_features, feature_cols)
all_features['actual_rank'] = y
all_features.head(20)



Saving features on ./results/test_results_2026_previous_data3/basic_features.csv
Saving temporal features on ./results/test_results_2026_previous_data3/temporal_features.csv
Saving all features on ./results/test_results_2026_previous_data3/all_features.csv


,animal,total_activity,mean_activity,std_activity,max_activity,min_activity,cv_activity,median_activity,max_median_ratio,activity_per_hour,...,night_bout_len_mean,night_bout_len_cv,night_transitions_per_hour,night_onset_latency_h,night_first_2h_frac,night_gini,night_peak_hour,night_activity_per_bout,night_day_intensity_ratio,actual_rank
animal_3,animal_3,1236.0,0.279955,0.550371,4.800000,-1.200000,1.965929,0.0,4.800000e+09,51.500000,...,0.638889,0.599701,0.782963,9.00,-0.012403,3.547242,5.0,1.343798,0.409731,3
animal_5,animal_5,1070.0,0.242356,0.475873,3.714286,-1.028571,1.963532,0.0,3.714286e+09,44.583333,...,0.711340,0.618770,0.703217,7.75,-0.032168,8.058770,6.0,0.574774,0.225111,5
animal_8,animal_8,2099.0,0.475425,0.926645,9.600000,-1.028571,1.949089,0.0,9.600000e+09,87.458333,...,0.693467,0.641862,0.721341,9.25,-0.015265,4.237123,5.0,1.135120,0.222627,7
animal_9,animal_9,2046.0,0.463420,0.846840,7.542857,-1.285714,1.827370,0.0,7.542857e+09,85.250000,...,0.624434,0.721260,0.801087,8.75,-0.022697,5.945514,5.0,0.710002,0.149274,6
animal_11,animal_11,2904.0,0.657758,1.085883,8.571429,-1.371429,1.650886,0.0,8.571429e+09,121.000000,...,0.570248,0.776352,0.877209,9.25,0.000000,0.000000,5.0,-0.038312,-0.005464,2
animal_12,animal_12,2585.0,0.585504,1.119429,12.342857,-1.285714,1.911906,0.0,1.234286e+10,107.708333,...,0.693467,0.697452,0.721341,7.00,-0.057842,14.091819,5.0,0.284565,0.039593,8
animal_13,animal_13,4497.0,1.018573,1.488940,9.600000,-1.457143,1.461790,0.2,4.800000e+01,187.375000,...,0.518797,0.674079,0.964205,8.75,-0.052173,11.955016,1.0,0.296142,0.026835,4
animal_15,animal_15,2251.0,0.509853,0.991756,8.542857,-1.542857,1.945181,0.0,8.542857e+09,93.791667,...,0.676471,0.689625,0.739465,8.25,0.000000,0.000000,5.0,-0.118063,-0.018909,1


In [87]:
output_correlation = f'{output_folder}/feature_correlations.csv'
corr_df = calculate_correlations(all_features, feature_cols, y, output_path=output_correlation)

Saving correlation on ./results/test_results_2026_new_data/feature_correlations.csv


In [31]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Diagnostic: feature heatmap sorted by true rank ─────────────────────────
# Shows z-scored feature values per animal (rows = animals sorted by rank 1→N)
# Features sorted by |ρ| with actual_rank. Ideal: a clear gradient top→bottom.

top_diag = 30   # how many top features to show
diag_feats = corr_df[corr_df.index.isin(feature_cols)].head(top_diag).index.tolist()

# Sort animals by actual rank
af_sorted = all_features.sort_values('actual_rank')
animal_labels = [f"{a} (#{int(r)})" for a, r in zip(af_sorted['animal'], af_sorted['actual_rank'])]

# Z-score matrix
from sklearn.preprocessing import StandardScaler
Z = StandardScaler().fit_transform(af_sorted[diag_feats].values)

# Sign-align so that positive z → higher rank (consistent visual direction)
for j, feat in enumerate(diag_feats):
    rho_sign = np.sign(corr_df.loc[feat, 'correlation'])
    if rho_sign < 0:
        Z[:, j] *= -1

fig_heat = go.Figure(go.Heatmap(
    z=Z,
    x=diag_feats,
    y=animal_labels,
    colorscale='RdBu',
    zmid=0,
    colorbar=dict(title='z (sign-aligned)'),
))
fig_heat.update_layout(
    title='Feature Heatmap — Animals sorted by Rank (top = most dominant)',
    height=max(400, 40 * len(af_sorted)),
    width=max(900, 30 * len(diag_feats)),
    xaxis=dict(tickangle=-45, tickfont=dict(size=9)),
    yaxis=dict(tickfont=dict(size=9)),
    plot_bgcolor='white', paper_bgcolor='white',
)
fig_heat.write_html(f"{output_folder}/feature_heatmap.html")
fig_heat.show()

# ── Scatter grid: top-6 features vs actual rank ──────────────────────────────
top6 = diag_feats[:6]
fig_scatter = make_subplots(rows=2, cols=3,
    subplot_titles=[f"{f}<br>ρ={corr_df.loc[f,'correlation']:+.3f}" for f in top6])

for idx, feat in enumerate(top6):
    r, c = divmod(idx, 3)
    x_vals = af_sorted['actual_rank'].tolist()
    y_vals = af_sorted[feat].tolist()
    fig_scatter.add_trace(go.Scatter(
        x=x_vals, y=y_vals, mode='markers+text',
        text=af_sorted['animal'].tolist(),
        textposition='top center', textfont=dict(size=7),
        marker=dict(size=9, color=x_vals, colorscale='Viridis', showscale=False),
        showlegend=False,
    ), row=r+1, col=c+1)
    fig_scatter.update_xaxes(title_text='True rank', row=r+1, col=c+1)
    fig_scatter.update_yaxes(title_text=feat, row=r+1, col=c+1)

fig_scatter.update_layout(
    title='Top Feature Scatter vs True Rank',
    height=600, width=1100,
    plot_bgcolor='white', paper_bgcolor='white',
)
fig_scatter.write_html(f"{output_folder}/feature_scatter_vs_rank.html")
fig_scatter.show()
print(f"Saved heatmap → {output_folder}/feature_heatmap.html")
print(f"Saved scatter → {output_folder}/feature_scatter_vs_rank.html")


Saved heatmap → ./results/test_results_2026_previous_data3/feature_heatmap.html
Saved scatter → ./results/test_results_2026_previous_data3/feature_scatter_vs_rank.html


In [95]:

# ── Build TOP_FEATURES dynamically from corr_df (cell 4 output) ─────────────
min_rho  = 0.20   # lowered threshold — cast wider net, search will still rank by |ρ|
top_n    = 30     # more candidates

top_corr = corr_df[corr_df.index.isin(feature_cols)].copy()
top_corr = top_corr[top_corr['correlation'].abs() >= min_rho].head(top_n)

TOP_FEATURES = {
    feat: (int(np.sign(row['correlation'])), round(abs(row['correlation']), 3))
    for feat, row in top_corr.iterrows()
}

print(f"TOP_FEATURES built from corr_df ({len(TOP_FEATURES)} features, |ρ| >= {min_rho}):")
print(f"{'Feature':<25} {'sign':>5}  {'|ρ|':>6}  {'p':>7}")
print("-" * 55)
for feat, (sign, rho_abs) in TOP_FEATURES.items():
    p = corr_df.loc[feat, 'p_value']
    sig = "**" if p < 0.01 else " *" if p < 0.05 else " ~" if p < 0.1 else ""
    print(f"  {feat:<23} {sign:>+5}  {rho_abs:>6.3f}  {p:>7.3f} {sig}")

# ── Exhaustive search ────────────────────────────────────────────────────────
available_feats = {f: v for f, v in TOP_FEATURES.items() if f in feature_cols}
feat_names = list(available_feats.keys())
y_true_cmp = all_features['actual_rank'].values

feat_vectors = {}
for feat, (sign, weight) in available_feats.items():
    col_idx = feature_cols.index(feat)
    feat_vectors[feat] = sign * X_scaled[:, col_idx]

best_results = []
total_combos = sum(len(list(combinations(feat_names, k))) for k in range(1, 6)) * 2

for k in range(1, 6):
    for combo in combinations(feat_names, k):
        composite_c = sum(feat_vectors[f] for f in combo)
        for polarity in [1, -1]:
            pred = rankdata(-polarity * composite_c).astype(int)
            rho_c, p_c = spearmanr(y_true_cmp, pred)
            mae_c = np.abs(y_true_cmp - pred).mean()
            if rho_c > 0:
                best_results.append((rho_c, p_c, mae_c, k, combo, polarity))

best_results.sort(key=lambda x: (-x[0], x[2]))

print(f"\nTop 20 feature combinations (out of {len(best_results)} valid, {total_combos} tested):\n")
print(f"{'ρ':>6} {'p':>7} {'MAE':>5}  {'k':>2}  Features")
print("-" * 90)
for rho_c, p_c, mae_c, k, combo, pol in best_results[:20]:
    sig = "**" if p_c < 0.01 else " *" if p_c < 0.05 else " ~" if p_c < 0.1 else "  "
    print(f"{rho_c:+.3f} {p_c:7.3f} {mae_c:5.2f}  {k:2d}  {', '.join(combo)} {sig}")

# ── Feature vote across top-10 results ──────────────────────────────────────
top_feats_pool = {}
for rho_c, p_c, mae_c, k, combo, pol in best_results[:10]:
    for f in combo:
        top_feats_pool[f] = top_feats_pool.get(f, 0) + rho_c

top_feats_ranked = sorted(top_feats_pool.items(), key=lambda x: -x[1])
print(f"\nMost frequent features in top-10 combos (by accumulated ρ):")
for feat, score in top_feats_ranked:
    sign, _ = available_feats[feat]
    print(f"  {feat:<25s}  score={score:.3f}  sign={'↑' if sign>0 else '↓'}")

# ── Apply the best combination ──────────────────────────────────────────────
best_rho, best_p, best_mae, best_k, best_combo, best_pol = best_results[0]
best_combo_str = " + ".join(best_combo)

print(f"\n→ Best combo (k={best_k}): {list(best_combo)}")
print(f"  ρ={best_rho:+.3f}, p={best_p:.3f}, MAE={best_mae:.2f}")

composite_best = best_pol * sum(feat_vectors[f] for f in best_combo)
pred_best = rankdata(-composite_best).astype(int)
all_features['predicted_rank_best'] = pred_best
all_features['composite_score_best'] = composite_best

comparison_best = all_features.set_index('animal')[['predicted_rank_best', 'composite_score_best']].copy()
comparison_best['true_rank'] = pd.Series(ranks)
comparison_best = comparison_best.sort_values('true_rank')
comparison_best['error'] = (comparison_best['predicted_rank_best'] - comparison_best['true_rank']).abs()

n = len(comparison_best)
print(f"\n  Spearman ρ      : {best_rho:+.3f}  (p={best_p:.3f})")
print(f"  MAE             : {best_mae:.2f} ranks")
print(f"  Exact match     : {(comparison_best['error']==0).sum()}/{n}  ({100*(comparison_best['error']==0).mean():.0f}%)")
print(f"  Within ±1 rank  : {(comparison_best['error']<=1).sum()}/{n}  ({100*(comparison_best['error']<=1).mean():.0f}%)")
print(f"  Within ±2 ranks : {(comparison_best['error']<=2).sum()}/{n}  ({100*(comparison_best['error']<=2).mean():.0f}%)")
print(f"\n{'Animal':<12} {'True':>6} {'Pred':>6} {'Error':>6}")
print("-" * 34)
for animal, row in comparison_best.iterrows():
    flag = " ✓" if row['error'] == 0 else (f" ~{int(row['error'])}" if row['error'] <= 2 else f" ✗{int(row['error'])}")
    print(f"{animal:<12} {int(row['true_rank']):>6} {int(row['predicted_rank_best']):>6} {int(row['error']):>6}{flag}")

# ── Save ranked output sorted by predicted rank ──────────────────────────────
ranked_output = comparison_best.copy().reset_index()
ranked_output = ranked_output.rename(columns={
    'predicted_rank_best': 'predicted_rank',
    'composite_score_best': 'composite_score',
    'error': 'absolute_error',
})
ranked_output['features_used'] = best_combo_str
ranked_output['spearman_rho']  = round(best_rho, 4)
ranked_output['p_value']       = round(best_p, 4)
ranked_output['mae']           = round(best_mae, 4)
ranked_output = ranked_output.sort_values('predicted_rank')

ranked_output_path = f"{output_folder}/best_combo_ranked.csv"
ranked_output.to_csv(ranked_output_path, index=False)

print(f"\n── Best combo ranked output (sorted by predicted rank) ──")
print(ranked_output[['animal', 'predicted_rank', 'true_rank', 'absolute_error', 'composite_score']].to_string(index=False))
print(f"\nSaved to {ranked_output_path}")

feature_rhos = corr_df.copy()
feature_rhos.rename(columns={'correlation': 'rho'}, inplace=True)
feature_rhos = feature_rhos.reset_index(names="feature")
feature_rhos.drop(columns=['p_value'], inplace=True)
feature_rhos = feature_rhos.set_index("feature")["rho"]



TOP_FEATURES built from corr_df (30 features, |ρ| >= 0.2):
Feature                    sign     |ρ|        p
-------------------------------------------------------
  rhythm_ratio_24_over_harm    +1   0.790    0.002 **
  power_24h                  +1   0.706    0.010  *
  autocorr_12h               -1   0.664    0.018  *
  power_12h                  -1   0.657    0.020  *
  cosinor_amplitude          +1   0.657    0.020  *
  median_activity            -1   0.647    0.023  *
  activity_kurtosis          -1   0.608    0.036  *
  high_activity_frac         +1   0.578    0.049  *
  activity_skew              -1   0.559    0.059  ~
  bout_len_cv                -1   0.545    0.067  ~
  max_activity               -1   0.545    0.067  ~
  peak_to_mean_ratio         -1   0.524    0.080  ~
  max_median_ratio           +1   0.517    0.085  ~
  cosinor_mesor              +1   0.517    0.085  ~
  night_mean_activity        -1   0.476    0.118 
  night_activity             -1   0.476    0.118 
  roll

In [72]:
feature_rhos_path = "./data/feature_rhos_new.csv"
feature_rhos_df = pd.read_csv(feature_rhos_path)

if {'feature', 'rho'}.issubset(feature_rhos_df.columns):
    feature_rhos_new = feature_rhos_df.set_index('feature')['rho']
else:
    raise ValueError(
        "`feature_rhos_previous.csv` must contain `feature` and `rho` columns. "
        "Recreate it with: feature_rhos.to_frame('rho').rename_axis('feature').to_csv(...)"
    )

feature_rhos_new.head()

feature
rhythm_ratio_24_over_harm    0.790210
power_24h                    0.706294
autocorr_12h                -0.664336
power_12h                   -0.657343
cosinor_amplitude            0.657343
Name: rho, dtype: float64

In [96]:
# k            : number of top features used by top-k methods
# best_feat_idx: feature name (or int index) for the single best feature proxy
# named_combo  : list of feature names for the exhaustive-search best combo

print(best_combo)
best_feat = None
if best_feat is None:
    print("Best feat is none, computing")
    best_feat = corr_df[corr_df.index.isin(feature_cols)]['correlation'].abs().idxmax()

if named_combo is None:
    print("Named combo is none, using computed")
    named_combo = list(best_combo) if 'best_combo' in dir() else None
    
print(best_feat)
print(named_combo)

#best_feat = 'rhythm_ratio_24_over_harm'
#named_combo = ['power_24h', 'high_activity_frac', 'activity_per_bout']
#best_feat = 'power_8h'


named_combo = ["power_24h", "short_gap_frac", "night_ibi_cv"]

feature_rhos, proxies_raw = build_all_proxies(
    all_features, feature_cols, X_scaled, y,
    k=3, best_feat_idx=best_feat,
    named_combo=named_combo,
    feature_rhos=feature_rhos

)



print(f"Best single feature: {best_feat}")
print('Top feature correlations (Spearman):')
print(feature_rhos.head(10))




proxy_output_path = f"{output_folder}/proxy_summary.csv"
proxy_rows, proxy_summary, results = evaluate_proxies(proxies_raw, all_features, y, verbose=True, output_path=proxy_output_path)

print("\nUnsupervised proxies vs rank (Spearman rho, MAE):")
for row in proxy_rows:
    print(f"{row['proxy']:<40s} rho={row['rho']: .3f}, MAE={row['mae']: .2f}, tau={row['kendall_tau']: .3f}")


('rhythm_ratio_24_over_harm', 'power_24h', 'peak_to_mean_ratio', 'max_median_ratio', 'cosinor_mesor')
Best feat is none, computing
Named combo is none, using computed
rhythm_ratio_24_over_harm
['rhythm_ratio_24_over_harm', 'power_24h', 'peak_to_mean_ratio', 'max_median_ratio', 'cosinor_mesor']
Using feature_rhos provided externally (e.g. from a reference dataset).
Best single feature: rhythm_ratio_24_over_harm
Top feature correlations (Spearman):
feature
rhythm_ratio_24_over_harm    0.790210
power_24h                    0.706294
autocorr_12h                -0.664336
power_12h                   -0.657343
cosinor_amplitude            0.657343
median_activity             -0.646554
activity_kurtosis           -0.608392
high_activity_frac           0.577552
activity_skew               -0.559441
bout_len_cv                 -0.545455
Name: rho, dtype: float64

Best feature (rhythm_ratio_24_over_harm):
  Predicted ranks are generated with: rankdata(scores_oriented, method="ordinal")
  Metrics:

In [97]:
actual_ranks = all_features['actual_rank'].values

for name,score in proxies_raw.items():
    print(score)
    ord_idx = np.argsort(score)
    print(ord_idx)
    ordered_animals = all_features.iloc[ord_idx]['animal'].tolist()
    print(name)
    pred_rank = rankdata(score, method='ordinal')
    print(ordered_animals)
    print(pred_rank)
    print(actual_ranks)

    metrics = rank_accuracy(actual_ranks, pred_rank)
    print(metrics)
    print()


[ 3.85705098 29.42850442 19.04186939 31.00691073  2.87386727 10.67778637
  7.3531821   0.72948246]
[7 4 0 6 5 2 1 3]
Best feature (rhythm_ratio_24_over_harm)
['animal_15', 'animal_11', 'animal_3', 'animal_13', 'animal_12', 'animal_8', 'animal_5', 'animal_9']
[3 7 6 8 2 5 4 1]
[3 5 7 6 2 8 4 1]
{'accuracy': 0.5, 'within_1': 0.625, 'within_2': 0.875, 'mae': 1.0, 'rho': 0.7857142857142858}

[-1.38499826  0.25127916  1.98637152  1.09579365 -1.41587769  2.47029338
 -0.15607532 -2.84678643]
[7 4 0 6 1 3 2 5]
Best combo (power_24h + short_gap_frac + night_ibi_cv)
['animal_15', 'animal_11', 'animal_3', 'animal_13', 'animal_5', 'animal_9', 'animal_8', 'animal_12']
[3 5 7 6 2 8 4 1]
[3 5 7 6 2 8 4 1]
{'accuracy': 1.0, 'within_1': 1.0, 'within_2': 1.0, 'mae': 0.0, 'rho': 1.0}

[-0.46166609  0.08375972  0.66212384  0.36526455 -0.47195923  0.82343113
 -0.05202511 -0.94892881]
[7 4 0 6 1 3 2 5]
Combo sign-aligned mean (power_24h + short_gap_frac + night_ibi_cv)
['animal_15', 'animal_11', 'animal_3',

##### Total activy per day tem a soma de todos os dias. Ë possivel ver isso claramente abrindo o arquivo collect data onde será visto a atividade por zt de 0.5 em 0.5, somando-se todos os valores da coluna o valor para cada dia será igual ao total day activity

In [163]:
output_corr = f"{output_folder}/feature_correlations.html"
fig_corr = plot_correlation(corr_df, output_corr)

male_comparison_output = f"{output_folder}/animal_comparison_all.html"
fig_activity = plot_animals_activity(animals_protocols, ranks, output_path=male_comparison_output)

output_path = f"{output_folder}/cross_correlation_matrix.html"
corr_sync, pval_sync, fig_cross = plot_cross_correlation(animals_protocols, ranks, output_path=output_path)

animal_1 in ranks, adding to plot
animal_2 in ranks, adding to plot
animal_3 in ranks, adding to plot
animal_4 in ranks, adding to plot
animal_5 in ranks, adding to plot
animal_6 in ranks, adding to plot
animal_7 in ranks, adding to plot
animal_8 in ranks, adding to plot
animal_9 in ranks, adding to plot
animal_10 in ranks, adding to plot
animal_11 in ranks, adding to plot
animal_12 in ranks, adding to plot



Average Activity Synchronization:
       animal  avg_correlation  actual_rank
5    animal_6         0.669877            1
9   animal_10         0.738562            2
10  animal_11         0.747010            3
11  animal_12         0.722566            4
6    animal_7         0.747960            5
7    animal_8         0.704673            6
8    animal_9         0.751670            7
1    animal_2         0.643215            8
0    animal_1         0.718421            9
3    animal_4         0.724619           10
2    animal_3         0.630318           11
4    animal_5         0.627037           12

Correlation between synchronization and rank: -0.462 (p=0.131)


In [164]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import math

# ── Build per-proxy result dataframes from proxy_rows + proxies_raw ─────────
# Replicate evaluate_proxies orientation + eval_ordering exactly:
#   1. orient scores so rho(scores, actual_rank) >= 0
#      → higher score = higher rank number = LESS dominant
#   2. rankdata(scores_oriented, method='ordinal')
#      → rank 1 = lowest score = MOST dominant ✓
actual_ranks = all_features['actual_rank'].values
animal_names = all_features['animal'].tolist()

# Sort index by true rank so scatter x-axis goes 1 → N
sort_idx = np.argsort(actual_ranks)
actual_ranks_sorted = actual_ranks[sort_idx]
animal_names_sorted = [animal_names[i] for i in sort_idx]

proxy_results = {}
for row in proxy_rows:
    name   = row['proxy']
    scores = proxies_raw[name]
    rho_dir = spearmanr(scores, actual_ranks).correlation

    # Mirror evaluate_proxies: orient so rho >= 0
    scores_oriented = scores if (rho_dir is None or rho_dir >= 0) else -scores
    # Mirror eval_ordering: rankdata with ordinal (no negation)
    pred_ranks = rankdata(scores_oriented, method='ordinal').astype(int)

    # Sort by true rank for the scatter plot
    pred_ranks_sorted = pred_ranks[sort_idx]
    errors_sorted     = np.abs(pred_ranks_sorted - actual_ranks_sorted)

    proxy_results[name] = {
        'animals':   animal_names_sorted,
        'true_rank': actual_ranks_sorted.tolist(),
        'pred_rank': pred_ranks_sorted.tolist(),
        'errors':    errors_sorted.tolist(),
        'rho':       row['rho'],
        'mae':       row['mae'],
        'within_1':  row['within_1'],
        'n':         len(actual_ranks),
    }

n_methods  = len(proxy_results)
n_animals  = len(animal_names)
diag_vals  = list(range(1, n_animals + 2))

# Sort methods by |ρ| descending so the best appear first
sorted_names = sorted(proxy_results.keys(),
                      key=lambda n: -abs(proxy_results[n]['rho']))

# ── Figure 1: Summary overview (ρ and MAE for all methods) ──────────────────
summary_df = pd.DataFrame(proxy_rows).sort_values('rho', key=abs, ascending=True)

fig_summary = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Spearman ρ with Actual Rank', 'Mean Absolute Error (MAE)'),
    horizontal_spacing=0.18
)

colors_rho = ['#2E86AB' if r >= 0 else '#E84855' for r in summary_df['rho']]

fig_summary.add_trace(go.Bar(
    x=summary_df['rho'], y=summary_df['proxy'], orientation='h',
    marker_color=colors_rho,
    text=[f"{r:+.3f}" for r in summary_df['rho']],
    textposition='outside',
    showlegend=False,
), row=1, col=1)

fig_summary.add_trace(go.Bar(
    x=summary_df['mae'], y=summary_df['proxy'], orientation='h',
    marker_color='#F18F01',
    text=[f"{m:.2f}" for m in summary_df['mae']],
    textposition='outside',
    showlegend=False,
), row=1, col=2)

fig_summary.add_vline(x=0, line_color='black', line_width=1, row=1, col=1)

fig_summary.update_layout(
    title=dict(text='Proxy Methods — Summary (ρ & MAE)', x=0.5),
    height=max(350, 38 * n_methods),
    width=1200,
    plot_bgcolor='white',
    paper_bgcolor='white',
    margin=dict(l=280),
)
fig_summary.update_xaxes(title_text='Spearman ρ', row=1, col=1, range=[-1.05, 1.05])
fig_summary.update_xaxes(title_text='MAE (ranks)', row=1, col=2)

fig_summary.write_html(f"{output_folder}/rank_evaluation_summary.html")
fig_summary.show()
print(f"Saved → {output_folder}/rank_evaluation_summary.html")

# ── Figure 2: Per-method grid — 2 methods per row, each with scatter + bar ──
METHODS_PER_ROW = 2
n_rows = math.ceil(n_methods / METHODS_PER_ROW)
v_spacing = min(0.06, round(0.9 / max(n_rows - 1, 1), 4)) if n_rows > 1 else 0.05

subplot_titles = []
padded = sorted_names + [None] * (n_rows * METHODS_PER_ROW - n_methods)

for i in range(n_rows):
    for j in range(METHODS_PER_ROW):
        idx = i * METHODS_PER_ROW + j
        name = padded[idx]
        if name is not None:
            r = proxy_results[name]
            subplot_titles += [
                f"{name}<br><sup>ρ={r['rho']:+.3f}, MAE={r['mae']:.2f}, W±1={r['within_1']*100:.0f}%</sup>",
                "Per-Animal Error",
            ]
        else:
            subplot_titles += ["", ""]

fig_grid = make_subplots(
    rows=n_rows, cols=4,
    subplot_titles=subplot_titles,
    horizontal_spacing=0.07,
    vertical_spacing=v_spacing,
    column_widths=[0.3, 0.2, 0.3, 0.2],
)

for i, name in enumerate(sorted_names):
    r           = proxy_results[name]
    grid_row    = i // METHODS_PER_ROW + 1
    pair_idx    = i %  METHODS_PER_ROW
    scatter_col = 1 + pair_idx * 2
    bar_col     = scatter_col + 1

    true_r  = r['true_rank']
    pred_r  = r['pred_rank']
    errors  = r['errors']
    animals = r['animals']

    err_colors = ['#2ecc71' if e == 0 else '#f39c12' if e <= 2 else '#e74c3c'
                  for e in errors]

    # Scatter: x=True Rank, y=Predicted Rank — perfect prediction lies on diagonal
    fig_grid.add_trace(go.Scatter(
        x=true_r, y=pred_r,
        mode='markers+text',
        text=animals,
        textposition='top center',
        textfont=dict(size=7),
        marker=dict(color=err_colors, size=10, line=dict(color='white', width=1)),
        showlegend=False,
    ), row=grid_row, col=scatter_col)

    fig_grid.add_trace(go.Scatter(
        x=diag_vals, y=diag_vals, mode='lines',
        line=dict(color='gray', dash='dash', width=1),
        showlegend=False,
    ), row=grid_row, col=scatter_col)

    fig_grid.add_trace(go.Scatter(
        x=diag_vals, y=[v + 1 for v in diag_vals], mode='lines',
        line=dict(color='lightgray', dash='dot', width=1), showlegend=False,
    ), row=grid_row, col=scatter_col)
    fig_grid.add_trace(go.Scatter(
        x=diag_vals, y=[v - 1 for v in diag_vals], mode='lines',
        line=dict(color='lightgray', dash='dot', width=1), showlegend=False,
    ), row=grid_row, col=scatter_col)

    fig_grid.update_xaxes(title_text="True Rank", row=grid_row, col=scatter_col)
    fig_grid.update_yaxes(title_text="Pred Rank", row=grid_row, col=scatter_col)

    # Bar: per-animal error sorted by true rank (rank 1 = most dominant on left)
    fig_grid.add_trace(go.Bar(
        x=animals, y=errors,
        marker_color=err_colors,
        text=[str(int(e)) for e in errors],
        textposition='outside',
        showlegend=False,
    ), row=grid_row, col=bar_col)

    fig_grid.update_xaxes(tickangle=-45, tickfont=dict(size=7), row=grid_row, col=bar_col)
    fig_grid.update_yaxes(title_text="Error", row=grid_row, col=bar_col)

fig_grid.update_layout(
    title=dict(text='Rank Evaluation — All Proxy Methods', x=0.5),
    height=350 * n_rows,
    width=1300,
    plot_bgcolor='white',
    paper_bgcolor='white',
    showlegend=False,
)

fig_grid.write_html(f"{output_folder}/rank_evaluation_all_methods.html")
fig_grid.show()
print(f"Saved → {output_folder}/rank_evaluation_all_methods.html")


Saved → ./results/test_results_2026_new_data/rank_evaluation_summary.html


Saved → ./results/test_results_2026_new_data/rank_evaluation_all_methods.html
